# Session 2: Building an LLM Survey Simulation Pipeline with QSTN

In this session, we create a whole pipeline for creating synthetic survey responses. We use [QSTN](https://github.com/dess-mannheim/QSTN), an open-source Python framework for constructing survey prompts, running LLM-based survey simulations, and parsing the resulting answers. By the end of the notebook, you will have a complete and reusable pipeline for creating synthetic survey responses.

Starting from interview-style persona descriptions, we define a questionnaire, generate simulated answers with several prompting strategies, and save the results for evaluation in Session 4.


## Installation

In [1]:
!pip install uv
!uv pip install --system vllm==0.24.0 --torch-backend=auto
!uv pip install qstn==0.5.2

Using Python 3.12.13 environment at: /usr
Checked 1 package in 581ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 853ms


### Apply fixes for google colab to run vllm

This cell is only necessary if you run this notebook in google colab. Otherwise you can skip this cell.

In [2]:
import os
import sys
import ctypes

# Fix CUDA 13 runtime
cu13_lib = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = cu13_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
ctypes.CDLL(f"{cu13_lib}/libcudart.so.13", mode=ctypes.RTLD_GLOBAL)

# Avoid vLLM V1 engine-core multiprocessing inside Colab/Jupyter.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"


# Give Jupyter stdout/stderr a fileno() so vLLM's suppress_stdout() does not crash.
class FilenoWrapper:
    def __init__(self, stream, fd):
        self._stream = stream
        self._fd = fd

    def fileno(self):
        return self._fd

    def write(self, data):
        return self._stream.write(data)

    def flush(self):
        return self._stream.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self._stream, name)


sys.stdout = FilenoWrapper(sys.stdout, 1)
sys.stderr = FilenoWrapper(sys.stderr, 2)

print("Colab/vLLM startup patches applied")

Colab/vLLM startup patches applied


## Imports

In [3]:
from copy import deepcopy

from datasets import load_dataset

from qstn.prompt_builder import LLMPrompt, generate_likert_options
from qstn.utilities import placeholder, create_one_dataframe
from qstn.survey_manager import (
    conduct_survey_battery,
    conduct_survey_sequential,
    conduct_survey_single_item,
)
from qstn.parser import parse_with_llm, to_dataframe
from qstn.inference import (
    JSONReasoningResponseGenerationMethod,
    JSONVerbalizedDistribution,
    JSONSingleResponseGenerationMethod,
)

from vllm import LLM

For demo purposes we will limit our simulation to just 20 survey respondents.

In [4]:
NUMBER_OF_PERSONAS_TO_SIMULATE = 20

## Start a local LLM

For this tutorial we will use a local `vllm` instance. QSTN also supports the OpenAI API specification.

We can define the class exactly the same way as without QSTN.

In [5]:
model_id = "Qwen/Qwen3-VL-2B-Instruct"

model = LLM(
    model=model_id, tensor_parallel_size=1, max_model_len=5000, max_num_seqs=200
)

INFO 07-24 16:53:55 [api_utils.py:273] non-default args: {'max_model_len': 5000, 'max_num_seqs': 200, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-VL-2B-Instruct'}
WARNING 07-24 16:53:56 [arg_utils.py:1590] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 07-24 16:53:58 [model.py:598] Resolved architecture: Qwen3VLForConditionalGeneration
WARNING 07-24 16:53:58 [model.py:2010] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 07-24 16:53:58 [model.py:2063] Casting torch.bfloat16 to torch.float16.
INFO 07-24 16:53:58 [model.py:1725] Using max model len 5000
INFO 07-24 16:53:58 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-24 16:53:58 [vllm.py:1006] Asynchronous scheduling is enabled.
INFO 07-24 16:53:58 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


INFO 07-24 16:54:32 [core.py:114] Initializing a V1 LLM engine (v0.24.0) with config: model='Qwen/Qwen3-VL-2B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen3-VL-2B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 07-24 16:54:38 [weight_utils.py:574] No model.safetensors.index.json found in remote.
INFO 07-24 16:54:38 [weight_utils.py:849] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 3.96 GiB. Available RAM: 8.66 GiB.
INFO 07-24 16:54:38 [weight_utils.py:872] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-24 16:54:52 [default_loader.py:430] Loading weights took 13.70 seconds
INFO 07-24 16:54:53 [gpu_model_runner.py:5255] Model loading took 4.31 GiB memory and 16.079056 seconds
INFO 07-24 16:54:53 [gpu_model_runner.py:6271] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
INFO 07-24 16:55:35 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/de517d788b/rank_0_0/backbone for vLLM's torch.compile
INFO 07-24 16:55:35 [backends.py:1148] Dynamo bytecode transform time: 3.82 s
INFO 07-24 16:55:38 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.099 s
INFO 07-24 16:55:38 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/c106c7419be55bb36a7e77247e0a8e1bd110257f8df34aaeb6d4cce5bb27b0be/rank_0_0/model
INFO 07-24 16:55:38 [monitor.py:53] torch.compile took 6.26 s in 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 44/44 [00:03<00:00, 11.30it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 28/28 [00:04<00:00,  5.85it/s]


INFO 07-24 16:55:54 [gpu_model_runner.py:6656] Graph capturing finished in 11 secs, took 0.37 GiB
INFO 07-24 16:55:54 [gpu_worker.py:667] CUDA graph pool memory: 0.37 GiB (actual), 0.42 GiB (estimated), difference: 0.05 GiB (13.1%).
INFO 07-24 16:55:54 [jit_monitor.py:60] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
INFO 07-24 16:55:56 [core.py:337] init engine (profile, create kv cache, warmup model) took 63.39 s (compilation: 6.26 s)


## What do we want to do?

Our goal is to simulate answers to two held-out items from the *General Social Survey* (GSS). For each respondent, the model receives an interview-style persona assembled from their other survey answers, but it does **not** see their true answers to the two target questions.

The held-out survey items are:

- `gss_182`: *Should government do more, or leave more to individuals and businesses?*
- `gss_379`: *Should immigration be increased or reduced?*

We will use the same respondents and questionnaire across several prompting and response-generation strategies. This gives us a controlled way to ask:

- How accurately can an LLM predict an individual response?
- Do the simulated answers reproduce the human population and subgroup distributions?
- Do they preserve the relationship between the two attitudes?
- Does one prompting strategy work better than the others?

The human answers remain our ground truth and will only be joined with the simulations during the evaluation in Session 4.


In [6]:
df_questionnaire = load_dataset(
    "dess-mannheim/ic2s2_tutorial", "questionnaire", split="train"
).to_pandas()
display(df_questionnaire)

,questionnaire_item_id,question_content
0,gss_182,"Should government do more, or leave more to in..."
1,gss_379,Should immigration be increased or reduced?


In [7]:
gss_182 = "gss_182"
gss_379 = "gss_379"

## Personas

Personas can be built in multiple ways. We will create the persona prompts based on the findings in [Prompt makes the Person(a)](https://aclanthology.org/2025.findings-emnlp.1261/) by Lutz et al. The persona could be introduced in any other way, for example, in a direct ("You are a ...") or from a third-person perspective ("Think of a ...").

Lutz et al. showed that *interview-style prompting* improved alignment in survey response prediction tasks. Thus, after loading the GSS survey data, we create interview-style prompts based on the individual responses of the survey respondents.

In [8]:
df_personas = load_dataset(
    "dess-mannheim/ic2s2_tutorial", "interviews", split="train"
).to_pandas()

In [9]:
df_personas.columns

Index(['respondent_id', 'sexual_orientation', 'family_origin', 'primary_race',
       'housing_tenure', 'household_type', 'family_generations_in_household',
       'partner_living_arrangement', 'workplace_sector', 'adults_in_household',
       'parent_of_child_in_household', 'religiosity',
       'standard_of_living_outlook', 'traditional_household_roles',
       'presidential_vote_2020', 'party_identification', 'political_views',
       'age_group', 'highest_degree', 'born_in_us', 'parents_born_in_us',
       'family_income', 'religious_preference', 'religious_service_attendance',
       'interview_prompt'],
      dtype='str')

In [10]:
print(df_personas.interview_prompt.iloc[0])

Q: Which of these best describes your sexual orientation?
A: Heterosexual or straight

Q: What is the first race you consider yourself to be?
A: Black or African American

Q: Do you (or your family) own your home, pay rent, or have some other arrangement?
A: Own or is buying

Q: What best describes the overall type of your household in terms of family structure and presence of children?
A: Cohabitating couple with children

Q: How many generations of your family live in your household?
A: Two generations, children

Q: Which best describes your current relationship status and living arrangement with a spouse or steady partner?
A: I am living as married and my partner and i together live in the same household

Q: How would you classify the type of place where you worked—manufacturing, wholesale trade, retail trade, or something else?
A: Other (agriculture, construction, service, government, etc.)

Q: How many adults live in your household in total?
A: 2

Q: To what extent do you consider

In [11]:
df_personas.interview_prompt.size

3309

## Prompting

QSTN allows you to configure a system prompt and a prompt.

Additionally, you can define the following placeholders, to dynamically change your system or prompt:

`placeholder.PROMPT_QUESTIONS`: Here the questions of your questionnaire will be asked.

`placeholder.PROMPT_OPTIONS`: If you want the model to give a certain option, you can give information about them here.

`placeholder.PROMPT_AUTOMATIC_OUTPUT_INSTRUCTIONS`: For different response generations, you can automatically define different output instructions.

`placeholder.QUESTION_CONTENT`: You can define just the content of your question and formulate your question yourself.

For now let's just define the placement of the questions and support automatic output instruction.


In [12]:
system_prompt = (
    f"You will see an interview of a person in a survey. "
    f"Your task is to predict the next answers of the survey respondent. "
    f"{placeholder.PROMPT_AUTOMATIC_OUTPUT_INSTRUCTIONS}"
)

# We create one prompt per persona:
prompts = []

for i, row in df_personas.iterrows():
    prompts.append(
        f"{row['interview_prompt']}\n\nPredict the answer to this question: {placeholder.PROMPT_QUESTIONS}"
    )

### LLMPrompt

`LLMPrompt` is the main class of QSTN, which allows you to setup modular prompts.
In general, it makes sense to create one `LLMPrompt` for each of your personas.

In [13]:
all_llm_prompts: list[LLMPrompt] = []

for prompt, respondent_id in zip(prompts, df_personas.respondent_id):
    llm_prompt = LLMPrompt(
        # Our questions
        questionnaire_source=df_questionnaire,
        # Name our questionnaire, so we can easily identify the persona afterwards
        questionnaire_name=respondent_id,
        # System prompt, prompt and seed
        system_prompt=system_prompt,
        prompt=prompt,
        seed=42,
    )
    all_llm_prompts.append(llm_prompt)

QSTN now automatically creates the prompts for you, with each LLMPrompt having one persona and automatically inserts the question at the requested place. Since we do not have specified any answer options so far, the prompt does not contain them.

In [14]:
example_prompt = all_llm_prompts[0]

print(str(example_prompt)[:365])
print('\n[…]\n')
print(str(example_prompt)[-275:])

=== gss_p_1 ===
=== SYSTEM_PROMPT ===
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. 
=== USER_PROMPT_WITH_ALL_QUESTIONS ===
Q: Which of these best describes your sexual orientation?
A: Heterosexual or straight

Q: What is the first race you consider yourself to be?
A: Black or African America

[…]

Q: What is your religious preference, if any?
A: Protestant

Q: How often do you attend religious services?
A: Every week

Predict the answer to this question: Should government do more, or leave more to individuals and businesses?
Should immigration be increased or reduced?


## Open-Ended Inference

We can directly generate answers, one persona & question at a time.

In [15]:
results = conduct_survey_single_item(
    model, all_llm_prompts[:NUMBER_OF_PERSONAS_TO_SIMULATE], max_tokens=5000
)

Running survey steps:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]

INFO 07-24 16:56:09 [hf.py:548] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.



Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-24 16:56:09 [jit_monitor.py:106] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.



Processed prompts: 100%|██████████| 20/20 [00:19<00:00,  1.03it/s, est. speed input: 485.41 toks/s, output: 271.63 toks/s]


Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:28<00:00,  1.43s/it, est. speed input: 326.68 toks/s, output: 299.21 toks/s]


We can now parse all the results to a pandas DataFrame with the method `to_dataframe`.

In [16]:
df_raw_answers = to_dataframe(results)

In [17]:
df_raw_answers.head()

,questionnaire_name,questionnaire_item_id,question,llm_response,logprobs,reasoning
0,gss_p_1,gss_182,"Should government do more, or leave more to in...","Based on the survey respondent's answers, the ...",None,None
1,gss_p_1,gss_379,Should immigration be increased or reduced?,"Based on the survey responses provided, the re...",None,None
2,gss_p_2,gss_182,"Should government do more, or leave more to in...","Based on the provided survey responses, it is ...",None,None
3,gss_p_2,gss_379,Should immigration be increased or reduced?,"Based on the provided information, it is not p...",None,None
4,gss_p_3,gss_182,"Should government do more, or leave more to in...","Based on the respondent's answers, which revea...",None,None


## Response Options

Often in surveys, due to workload balances we request participants to fill out answers on a scale, instead of giving full text answers. With QSTN we can easily add them and modify them to our wishes.

In the original GSS data, we have two 5-item likert scales, one with two endpoints and one with 5 exact points.

**gss_182 — role of government:**

1. Government should do more to solve the country’s problems.
2.
3.  
4.   
5. Government is doing too many things that should be left to individuals and private businesses.

**gss_379 — desired immigration level:**

1. Increased a lot  
2. Increased a little  
3. Remain the same  
4. Reduced a little  
5. Reduced a lot

Let's add them to our `LLMPrompt`.

In [18]:
government_response_option = generate_likert_options(
    # scale size
    n=5,
    # scale texts
    answer_texts=[
        "Government should do more to solve the country’s problems",
        "Government is doing too many things that should be left to individuals and private businesses",
    ],
    # We only label the endpoints
    only_from_to_scale=True,
    # How the options should be presented to the model
    scale_prompt_template="Your response should range on the scale from {start} to {end}",
)

immigration_response_option = generate_likert_options(
    # scale size
    n=5,
    # scale texts
    answer_texts=[
        "Increased a lot",
        "Increased a little",
        "Remain the same",
        "Reduced a little",
        "Reduced a lot",
    ],
    # How the options should be presented to the model
    list_prompt_template="Your response should clearly be one of these options {options}",
)

both_options = {
    gss_182: government_response_option,
    gss_379: immigration_response_option,
}

In [19]:
llm_prompts_with_options = deepcopy(all_llm_prompts)

for llm_prompt in llm_prompts_with_options:
    llm_prompt = llm_prompt.prepare_prompt(
        question_stem=f"{placeholder.QUESTION_CONTENT}\n{placeholder.PROMPT_OPTIONS}",
        answer_options=both_options,
    )

example_prompt = llm_prompts_with_options[0]

print(str(example_prompt)[:365])
print('\n[…]\n')
print(str(example_prompt)[-512:])

=== gss_p_1 ===
=== SYSTEM_PROMPT ===
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. 
=== USER_PROMPT_WITH_ALL_QUESTIONS ===
Q: Which of these best describes your sexual orientation?
A: Heterosexual or straight

Q: What is the first race you consider yourself to be?
A: Black or African America

[…]

Predict the answer to this question: Should government do more, or leave more to individuals and businesses?
Your response should range on the scale from 1: Government should do more to solve the country’s problems to 5: Government is doing too many things that should be left to individuals and private businesses
Should immigration be increased or reduced?
Your response should clearly be one of these options 1: Increased a lot, 2: Increased a little, 3: Remain the same, 4: Reduced a little, 5: Reduced a lot


In [20]:
results = conduct_survey_single_item(
    model, llm_prompts_with_options[:NUMBER_OF_PERSONAS_TO_SIMULATE], max_tokens=100
)

Running survey steps:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:01<00:00, 17.23it/s, est. speed input: 8829.16 toks/s, output: 310.64 toks/s]


Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:00<00:00, 34.26it/s, est. speed input: 17553.01 toks/s, output: 207.07 toks/s]


In [21]:
df_raw_likert_answers = to_dataframe(results)

In [22]:
df_raw_likert_answers.head()

,questionnaire_name,questionnaire_item_id,question,llm_response,logprobs,reasoning
0,gss_p_1,gss_182,"Should government do more, or leave more to in...",4: Government is doing too many things that sh...,None,None
1,gss_p_1,gss_379,Should immigration be increased or reduced?\nY...,3: Remain the same,None,None
2,gss_p_2,gss_182,"Should government do more, or leave more to in...",3: Government is doing too many things that sh...,None,None
3,gss_p_2,gss_379,Should immigration be increased or reduced?\nY...,3: Remain the same,None,None
4,gss_p_3,gss_182,"Should government do more, or leave more to in...",4: Government is doing too many things that sh...,None,None


## Response Generation Methods

If we want to make sure that the LLM answers with the exact statement we can also use structured output to restrict the tokens that the model can generate.

In [23]:
reasoning_rgm = JSONReasoningResponseGenerationMethod()

# With a dedicated thinking model you can also use
single_rgm = JSONSingleResponseGenerationMethod()


both_options[gss_182].response_generation_method = reasoning_rgm

both_options[gss_379].response_generation_method = reasoning_rgm

llm_prompts_structured = deepcopy(llm_prompts_with_options)

for llm_prompt in llm_prompts_structured:
    llm_prompt.prepare_prompt(answer_options=both_options)

 Since we have `placeholder.PROMPT_AUTOMATIC_OUTPUT_INSTRUCTIONS` in our prompt, the instructions get updated automatically.

In [24]:
sys_government, prompt_government = llm_prompts_structured[
    0
].get_prompt_for_questionnaire_type(item_id=gss_182)

print("=== System Prompt ===")
print(sys_government)
print("=== Prompt ===")
print('[…]\n', str(prompt_government)[-377:])

=== System Prompt ===
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. You always reason about the possible answer options first.
You respond with your reasoning and the most probable answer option in the following JSON format:
{
  "reasoning": "your reasoning about the answer options",
  "answer": "Your response should range on the scale from 1: Government should do more to solve the country’s problems to 5: Government is doing too many things that should be left to individuals and private businesses"
}
=== Prompt ===
[…]
 
Q: How often do you attend religious services?
A: Every week

Predict the answer to this question: Should government do more, or leave more to individuals and businesses?
Your response should range on the scale from 1: Government should do more to solve the country’s problems to 5: Government is doing too many things that should be left to individuals and private businesses


In [25]:
reasoning_result = conduct_survey_single_item(
    model, llm_prompts_structured[:NUMBER_OF_PERSONAS_TO_SIMULATE], max_tokens=3000
)

Running survey steps:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-24 16:57:00 [jit_monitor.py:106] Triton kernel JIT compilation during inference: apply_token_bitmask_inplace_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.



Processed prompts: 100%|██████████| 20/20 [01:38<00:00,  4.93s/it, est. speed input: 121.42 toks/s, output: 107.22 toks/s]


Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:19<00:00,  1.01it/s, est. speed input: 604.25 toks/s, output: 221.72 toks/s]


In [26]:
reasoning_df = to_dataframe(reasoning_result)

In [27]:
reasoning_df.head()

,questionnaire_name,questionnaire_item_id,question,reasoning,answer
0,gss_p_1,gss_182,"Should government do more, or leave more to in...","Based on the survey responses, the respondent ...",3
1,gss_p_1,gss_379,Should immigration be increased or reduced?\nY...,"Based on the survey responses provided, the re...",2: Increased a little
2,gss_p_2,gss_182,"Should government do more, or leave more to in...",The respondent's answers indicate a strong sup...,2
3,gss_p_2,gss_379,Should immigration be increased or reduced?\nY...,The respondent's answers indicate a strong pre...,4: Reduced a little
4,gss_p_3,gss_182,"Should government do more, or leave more to in...","The survey respondent is a 60–74-year-old, Whi...",1: Government should do more to solve the coun...


There is growing evidence that verbalized sampling or verbalized distrbution improve alignment with human responses ([Ahnert et al.](https://arxiv.org/abs/2510.11586), [Meister et al.](https://arxiv.org/abs/2411.05403)).
Here we ask the model to give a probability for each answer, which leads to more variety in the final output.

In [28]:
verbalized_distribution = JSONVerbalizedDistribution(output_index_only=False)

both_options[gss_182].response_generation_method = verbalized_distribution
both_options[gss_379].response_generation_method = verbalized_distribution

# Adjust the prompt in this case
both_options[
    gss_182
].scale_prompt_template = "Give probabilites for each of these options on scale from {start} to {end}, Each of them should be between 0 and 1 and all 5 should sum up to 1."

both_options[
    gss_379
].list_prompt_template = "Give probabilites for each of these {options}, Each of them should be between 0 and 1 and all 5 should sum up to 1."

llm_prompts_verbalized = deepcopy(llm_prompts_with_options)

for llm_prompt in llm_prompts_verbalized:
    llm_prompt.prepare_prompt(answer_options=both_options)

In [29]:
sys_government, prompt_government = llm_prompts_verbalized[
    0
].get_prompt_for_questionnaire_type(item_id=gss_379)

print("=== System Prompt ===")
print(sys_government)
print("=== Prompt ===")
print('[…]\n', str(prompt_government)[-350:])

=== System Prompt ===
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. You only respond with a probability for each answer option in the following JSON format:
{
  "1: Increased a lot": "probability for: 1: Increased a lot",
  "2: Increased a little": "probability for: 2: Increased a little",
  "3: Remain the same": "probability for: 3: Remain the same",
  "4: Reduced a little": "probability for: 4: Reduced a little",
  "5: Reduced a lot": "probability for: 5: Reduced a lot"
}
=== Prompt ===
[…]
 
Q: How often do you attend religious services?
A: Every week

Predict the answer to this question: Should immigration be increased or reduced?
Give probabilites for each of these 1: Increased a lot, 2: Increased a little, 3: Remain the same, 4: Reduced a little, 5: Reduced a lot, Each of them should be between 0 and 1 and all 5 should sum up to 1.


In [30]:
verbalized_distribution_result = conduct_survey_single_item(
    model, llm_prompts_verbalized[:NUMBER_OF_PERSONAS_TO_SIMULATE], max_tokens=1000
)

Running survey steps:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:07<00:00,  2.77it/s, est. speed input: 1841.22 toks/s, output: 211.98 toks/s]


Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:06<00:00,  3.08it/s, est. speed input: 1974.81 toks/s, output: 209.61 toks/s]


In [31]:
verbalized_df = to_dataframe(verbalized_distribution_result)

QSTN automatically parses the output into a dataframe with a column that contains the percentages for each answer option.

In [32]:
verbalized_df.head()

,questionnaire_name,questionnaire_item_id,question,1: Government should do more to solve the country’s problems,2,3,4,5: Government is doing too many things that should be left to individuals and private businesses,1: Increased a lot,2: Increased a little,3: Remain the same,4: Reduced a little,5: Reduced a lot
0,gss_p_1,gss_182,"Should government do more, or leave more to in...",0.6,0.3,0.1,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,gss_p_1,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.4,0.3,0.2,0.1,0.1
2,gss_p_2,gss_182,"Should government do more, or leave more to in...",0.45,0.35,0.15,0.05,0.05,NaN,NaN,NaN,NaN,NaN
3,gss_p_2,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.3,0.4,0.2,0.1,0.2
4,gss_p_3,gss_182,"Should government do more, or leave more to in...",0.6,0.3,0.1,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## Keeping previous questions in context

Very often it can make sense to keep the previous answers in context, especially if you have questions that correlate with each other, and you want the LLM to model this correlation as well.

These methods can improve alignment with human survey answers and in the case of battery mode, even save GPU time, as we have shown in our [QSTN Paper](https://aclanthology.org/2026.eacl-demo.37/) (Kreutner et al.).

In practice, when using `QSTN` you just have to exchange the `conduct_survey_single_item` with the corresponding method:


`conduct_survey_sequential` asks every question in a sequential chat format. After the first question the LLM responds, then the next question gets asked.

In [33]:
results_sequential = conduct_survey_sequential(
    model,
    llm_prompts=llm_prompts_verbalized[:NUMBER_OF_PERSONAS_TO_SIMULATE],
    max_tokens=1000,
    print_conversation=True,
    number_of_printed_conversations=1,
)

Running survey steps:   0%|          | 0/2 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:04<00:00,  4.25it/s, est. speed input: 2822.17 toks/s, output: 324.91 toks/s]

--- Conversation ---
-- System Message --
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. You only respond with a probability for each answer option in the following JSON format:
{
  "1: Government should do more to solve the country’s problems": "probability for: 1: Government should do more to solve the country’s problems",
  "2": "probability for: 2",
  "3": "probability for: 3",
  "4": "probability for: 4",
  "5: Government is doing too many things that should be left to individuals and private businesses": "probability for: 5: Government is doing too many things that should be left to individuals and private businesses"
}
-- User Message --
Q: Which of these best describes your sexual orientation?
A: Heterosexual or straight

Q: What is the first race you consider yourself to be?
A: Black or African American

Q: Do you (or your family) own your home, pay rent, or have some other arrangement?
A: Own or is buying


Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:06<00:00,  3.15it/s, est. speed input: 2536.06 toks/s, output: 217.74 toks/s]

--- Conversation ---
-- System Message --
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. You only respond with a probability for each answer option in the following JSON format:
{
  "1: Increased a lot": "probability for: 1: Increased a lot",
  "2: Increased a little": "probability for: 2: Increased a little",
  "3: Remain the same": "probability for: 3: Remain the same",
  "4: Reduced a little": "probability for: 4: Reduced a little",
  "5: Reduced a lot": "probability for: 5: Reduced a lot"
}
-- User Message --
Q: Which of these best describes your sexual orientation?
A: Heterosexual or straight

Q: What is the first race you consider yourself to be?
A: Black or African American

Q: Do you (or your family) own your home, pay rent, or have some other arrangement?
A: Own or is buying

Q: What best describes the overall type of your household in terms of family structure and presence of children?
A: Cohabitating coup

In [34]:
df_sequential = to_dataframe(results_sequential)

In [35]:
df_sequential.head()

,questionnaire_name,questionnaire_item_id,question,1: Government should do more to solve the country’s problems,2,3,4,5: Government is doing too many things that should be left to individuals and private businesses,1: Increased a lot,2: Increased a little,3: Remain the same,4: Reduced a little,5: Reduced a lot
0,gss_p_1,gss_182,"Should government do more, or leave more to in...",0.6,0.3,0.1,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,gss_p_1,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.4,0.3,0.2,0.1,0.0
2,gss_p_2,gss_182,"Should government do more, or leave more to in...",0.45,0.35,0.15,0.05,0.05,NaN,NaN,NaN,NaN,NaN
3,gss_p_2,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.55,0.3,0.1,0.05,0.05
4,gss_p_3,gss_182,"Should government do more, or leave more to in...",0.6,0.3,0.1,0.0,0.0,NaN,NaN,NaN,NaN,NaN


Battery asks all questions in a single prompt and the LLM should respond to all questions in one response.

In [36]:
results_battery = conduct_survey_battery(
    model,
    llm_prompts=llm_prompts_verbalized[:NUMBER_OF_PERSONAS_TO_SIMULATE],
    max_tokens=1000,
    print_conversation=True,
    number_of_printed_conversations=1,
)

Running survey steps:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/20 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 20/20 [00:15<00:00,  1.26it/s, est. speed input: 1088.35 toks/s, output: 217.45 toks/s]

--- Conversation ---
-- System Message --
You will see an interview of a person in a survey. Your task is to predict the next answers of the survey respondent. You only respond with a probability for each answer option in the following JSON format:
{
  "Should government do more, or leave more to individuals and businesses?": {
    "1: Government should do more to solve the country’s problems": "probability for: 1: Government should do more to solve the country’s problems",
    "2": "probability for: 2",
    "3": "probability for: 3",
    "4": "probability for: 4",
    "5: Government is doing too many things that should be left to individuals and private businesses": "probability for: 5: Government is doing too many things that should be left to individuals and private businesses"
  },
  "Should immigration be increased or reduced?": {
    "1: Increased a lot": "probability for: 1: Increased a lot",
    "2: Increased a little": "probability for: 2: Increased a little",
    "3: Remain t

In [37]:
df_battery = to_dataframe(results_battery)

In [38]:
df_battery.head()

,questionnaire_name,questionnaire_item_id,question,1: Government should do more to solve the country’s problems,2,3,4,5: Government is doing too many things that should be left to individuals and private businesses,1: Increased a lot,2: Increased a little,3: Remain the same,4: Reduced a little,5: Reduced a lot
0,gss_p_1,gss_182,"Should government do more, or leave more to in...",0.4,0.3,0.2,0.1,0.0,NaN,NaN,NaN,NaN,NaN
1,gss_p_1,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.2,0.1,0.5,0.2,0.1
2,gss_p_2,gss_182,"Should government do more, or leave more to in...",0.3,0.4,0.2,0.1,0.0,NaN,NaN,NaN,NaN,NaN
3,gss_p_2,gss_379,Should immigration be increased or reduced?\nG...,NaN,NaN,NaN,NaN,NaN,0.1,0.2,0.5,0.2,0.0
4,gss_p_3,gss_182,"Should government do more, or leave more to in...",0.4,0.3,0.2,0.1,0.0,NaN,NaN,NaN,NaN,NaN


## Evaluation

We will evaluate the results in session 4. For now we save all the DataFrames as csvs.

In [39]:
from pathlib import Path

RESULT_DIR = Path("tutorial_data") / model_id
RESULT_DIR.mkdir(parents=True, exist_ok=True)

num_participants = NUMBER_OF_PERSONAS_TO_SIMULATE


def save_result(df, method):
    df.to_csv(RESULT_DIR / f"result_{method}_{num_participants}.csv", index=False)

save_result(reasoning_df, "reasoning")
save_result(verbalized_df, "verbalized")
save_result(df_sequential, "sequential")
save_result(df_battery, "battery")